# 04. Bunching Detection

**Scope of this notebook:** determine whether two vehicles on the same route, traveling the same direction, are running too close together, which is the second of the two MVP metrics named in the README.

**Prerequisites established in prior notebooks:** dwell/layover classification and silent-vehicle threshold (01), trusted `stop_id`/`current_stop_sequence` fields (02), schedule deviation with arrival-vs-departure semantics and first-arrival collapsing (03).


## A. Do We Have What We Need? Researching the Direction of Travel

**Question:** Two vehicles on the same `route_id` can be close together for two very different reasons: 
- They're bunched (same direction, one has caught up to the other)
- They're just passing each other going opposite ways on a bidirectional route.

Flagging the second case as "bunching" would be an error. Do we currently have enough information to tell them apart?

**Method:** `route_id` + `trip_id` alone don't reliably encode direction: GTFS's `stop_sequence` numbering is trip-specific and doesn't provide a comparable axis across different trips. GTFS does define this properly via `direction_id`, in two places: on the real-time
`TripDescriptor` (decoded by `decoder.ts` already, but not currently selected into the published Kafka message by `validator.ts`), and separately as a static column on `trips.txt`, keyed by `trip_id`.

This exploration doesn't require a Node-side pipeline change to use today: every `trip_id` already flowing through Kafka can be joined against `trips.txt` directly in this notebook. Adding `direction_id` to `validator.ts`'s output is still worth doing eventually (fewer joins needed downstream, and it'd be available for the live Express API without a static-file dependency), but it's not a blocker here. It will be implemented in the production code.

In [1]:
import pandas as pd
import duckdb

GTFS_STATIC_PATH = '../gtfs_static/MBTA_GTFS'
AGENCY_TZ = 'America/New_York'

df_deduped = pd.read_parquet('telemetry_sample_N1.parquet')
df_deduped['timestamp'] = pd.to_datetime(df_deduped['timestamp'], utc=True)
df_deduped['timestamp_eastern'] = df_deduped['timestamp'].dt.tz_convert(AGENCY_TZ)

query_direction = f"""
    SELECT
        p.vehicle_id,
        p.trip_id,
        p.route_id,
        p.timestamp_eastern,
        p.current_status,
        p.lat,
        p.lon,
        t.direction_id
    FROM df_deduped AS p
    LEFT JOIN read_csv_auto('{GTFS_STATIC_PATH}/trips.txt', types={{'trip_id': 'VARCHAR'}}, ignore_errors=true) AS t
        ON p.trip_id = t.trip_id
"""

df_with_direction = duckdb.sql(query_direction).df()
df_with_direction['timestamp_eastern'] = df_with_direction['timestamp_eastern'].dt.tz_convert(AGENCY_TZ)

resolved_pct = df_with_direction['direction_id'].notna().mean() * 100
print(f"Pings with a resolved direction_id: {resolved_pct:.1f}%")
df_with_direction.head()


Pings with a resolved direction_id: 98.5%


,vehicle_id,trip_id,route_id,timestamp_eastern,current_status,lat,lon,direction_id
0,y1776,76676118,1,2026-08-18 19:29:00-04:00,STOPPED_AT,42.369987,-71.112877,0
1,y1785,76676122,1,2026-08-18 19:29:57-04:00,STOPPED_AT,42.359459,-71.093811,0
2,y1897,76676123,1,2026-08-18 19:28:58-04:00,STOPPED_AT,42.329811,-71.083519,0
3,y1875,76676126,1,2026-08-18 19:30:00-04:00,IN_TRANSIT_TO,42.336163,-71.076607,0
4,y1897,76676407,1,2026-08-18 19:24:27-04:00,IN_TRANSIT_TO,42.329819,-71.084229,1


**Result**: 98.5% of pings resolve a direction_id via the static join, which is actually consistent with notebook 02's ~99.5% stop-resolution rate, and the small remainder is almost certainly the same Shuttle-Generic*/no-schedule trips already scoped out there.

**Decision:** Direction resolves cleanly via a pure trips.txt join, no Node-side change required for this notebook. Still worth adding direction_id to validator.ts's output as a follow-up, since it'd let the live serving API compute bunching without a trips.txt dependency at request time, but it's not blocking anything in this notebook.

## B. A Formal Definition of Bunching: Candidate generation

**Question:** What, precisely, counts as a bunching event? A vague 'too close together' is neither implementable nor measurable: a single close GPS ping could just be noise, two vehicles at a shared intersection, or a momentary artifact rather than genuine bunching.

**Proposed definition:** To be validated, not assumed correct, by the sensitivity analysis in Section C:

Two vehicles are BUNCHED if, at the same approximate moment:

```
>  same route_id
>  same direction_id   (rules out opposite-direction pass-bys, per Section A)
>  straight-line distance <= DISTANCE_THRESHOLD_METERS
>  this proximity persists for >= MIN_CONSECUTIVE_OBSERVATIONS in a row
```
*Rules out a single noisy/momentary close ping*

**Method:** Compare every pair of same-route, same-direction vehicles at each polling snapshot (bucketed to the poll interval, so we're comparing "the same moment" across vehicles), using the same equirectangular meters approximation validated in notebooks 01/02. A pair is a bunching *candidate* at a given moment if within threshold; it's a bunching *event* only once persistence is checked in Section C.

In [2]:
POLL_INTERVAL_SECONDS = 15  # matches BASE_INTERVAL_MS in poller.ts

# drop any row where selected fields gave any missing value
df_bunch = df_with_direction.dropna(subset=['direction_id', 'lat', 'lon']).copy()
# floors timestamps to a 15 seconds interval: 10:00:01 -> 10:00:00
df_bunch['time_bucket'] = df_bunch['timestamp_eastern'].dt.floor(f'{POLL_INTERVAL_SECONDS}s')

# this doesn't produce hundreds of thousands of pairs because of the filters applied at the 
# end of the query, and omit too the redundancy of pairs. e.g. pair (a,b) doesn't generate a pair (b,a), or (a,a)
query_pairs = """

    -- Prepare the position data used for the self-join.
    -- Keep only the columns needed to identify and compare vehicles.
    WITH pos AS (
        SELECT vehicle_id, route_id, direction_id, time_bucket, lat, lon
        FROM df_bunch
    )
    SELECT
    
    -- Identify when and where the vehicle pair was observed.
        a.time_bucket,
        a.route_id,
        a.direction_id,
        
    -- Store the two vehicles forming the pair.
    -- "a" and "b" are two aliases of the same "pos" dataset.
        a.vehicle_id AS vehicle_a,
        b.vehicle_id AS vehicle_b,

    -- Calculate the approximate straight-line distance between
-- the two vehicles in meters using the equirectangular approximation
-- Latitude difference:
-- 1 degree of latitude ≈ 111,320 meters
        SQRT(
            POW((a.lat - b.lat) * 111320, 2) +
            POW((a.lon - b.lon) * 111320 * COS(RADIANS(a.lat)), 2)
        ) AS distance_meters

    -- SELF-JOIN:
    -- Compare the position dataset against itself so that we can
    -- create pairs of vehicles.
    FROM pos a
    JOIN pos b

    -- Same route, direction, time bucket and
    -- keeping each vehicle pair only once and prevent a vehicle
    -- from being paired with itself.
        ON a.route_id = b.route_id
        AND a.direction_id = b.direction_id
        AND a.time_bucket = b.time_bucket
        AND a.vehicle_id < b.vehicle_id
"""

df_pairs = duckdb.sql(query_pairs).df()
print(f"Same-route, same-direction, same-moment vehicle pairs evaluated: {len(df_pairs)}")
print()
print(df_pairs['distance_meters'].describe())


Same-route, same-direction, same-moment vehicle pairs evaluated: 12966

count    12966.000000
mean      4593.911988
std       6398.154545
min          1.788549
25%       1951.432766
50%       3292.786748
75%       5019.746600
max      91780.442204
Name: distance_meters, dtype: float64


**Result:** 12,966 same-route/same-direction/same-moment pairs evaluated. Mean distance 4,594m, median 3,293m, max 91,780m. The typical pair is over 3km apart, which is not close at all.

Worth stating explicitly in the notebook, since the number looks alarming out of context: this is expected, not a bug. The self-join compares every same-route, same-direction vehicle pair at each moment, so most of those pairs are just two vehicles spread out along a long route, nowhere near each other. This distribution is the full background population; genuine bunching candidates are the rare, near-zero left tail (min: 1.79m), not the typical case. That the median sits at ~3.3km is actually a good sanity check, because it confirms bunching is actually rare.

## C. Sensitivity Analysis: Bunching detection and Persistence

**Question:** How sensitive is the number of flagged bunching events to the exact distance and persistence values chosen? A single arbitrarily-picked threshold ("60 seconds/200 meters seemed reasonable") is a weak justification, and it is not backed by any real information; showing how the count changes across a range makes the final choice defensible and reproducible.

**Method:** Sweep both the distance threshold and the persistence requirement independently, and report the resulting event count at each combination.

In [3]:
def count_bunching_events(pairs_df, distance_threshold_m, min_consecutive):
    """Count distinct bunching EVENTS (not just candidate snapshots): a run of
    >= min_consecutive consecutive time_buckets where the same vehicle pair stays within
    distance_threshold_m counts as ONE event, not one per bucket."""
    
    # Take only the pair of buses that are closer than distance_threshold
    close = pairs_df[pairs_df['distance_meters'] <= distance_threshold_m].copy()
    if close.empty:
        return 0

    # Sorting pairs in chronological order, and generate an id. e.g. y1234_y5678
    close = close.sort_values(['vehicle_a', 'vehicle_b', 'time_bucket'])
    close['pair_id'] = close['vehicle_a'] + '_' + close['vehicle_b']

    # Chronologically sort EACH pair, giving them a rank number, chosing the first record with time_bucket 
    close['bucket_rank'] = close.groupby('pair_id')['time_bucket'].rank(method='first')

    # "Islands and Gaps" trick: 
    # If pings are truly consecutive, subtracting (rank * 15 seconds) from their timestamp 
    # will result in the exact same "base time" for the entire streak, for example,
    # 10:00:00-1*15=09:59:45; 10:00:15-2*15=09:59:45, etc.
    # A gap in time will shift this base time, breaking the streak.
    close['expected_if_consecutive'] = (
        close['time_bucket'] - pd.to_timedelta(close['bucket_rank'] * POLL_INTERVAL_SECONDS, unit='s')
    )

    # Assign a unique ID to each continuous run (streak) by grouping the vehicle pair and their shared base time
    run_id = close.groupby(['pair_id', 'expected_if_consecutive']).ngroup()

    # Count how many pings (rows) exist inside each continuous run.
    run_lengths = close.groupby(run_id).size()

    # Returns the amount of records that had a streak >= min_consecutive
    return int((run_lengths >= min_consecutive).sum())

distance_thresholds = [50, 100, 200, 300, 500]
persistence_options = [1, 2, 3, 4]

# Test the function 20 times to get a threshold (50 meters with 1 ping, 50 meters with 2 pings, etc..)
sweep = pd.DataFrame(
    [
        {
            'distance_threshold_m': d,
            'min_consecutive_polls': p,
            'bunching_events': count_bunching_events(df_pairs, d, p),
        }
        for d in distance_thresholds
        for p in persistence_options
    ]
)

sweep.pivot(index='distance_threshold_m', columns='min_consecutive_polls', values='bunching_events')

min_consecutive_polls,1,2,3,4
distance_threshold_m,,,,
50,90,45,22,14
100,131,71,36,21
200,164,90,58,30
300,200,111,73,41
500,282,161,106,57


**Result:**  The sensitivity analysis did not reveal a clear mathematical elbow. Event counts changed smoothly as thresholds became more permissive. Therefore, 100 meters is selected as an operationally interpretable spatial threshold and 2 consecutive observations as a minimal persistence requirement to eliminate isolated proximity observations. The choice is operationally anchored rather than mathematically optimized

**Decision:** Since there is no sheer mathematical inflection point, the thresholds are anchored operationally:

- **DISTANCE_THRESHOLD_METERS = 100:** Roughly 5-8 bus lengths. Close enough that a passenger would visually perceive them as "bunched".

- **MIN_CONSECUTIVE_OBSERVATIONS = 2:** Requires ~30 seconds of sustained proximity, effectively filtering out single-ping GPS noise or momentary intersection pass-bys.

The sweep's role confirms there is no erratic spike immediately around this choice (71 events in this sample). 

*(Known Limitation: Simple 15s time-bucketing might occasionally split a true bunching event across boundaries due to the ~16s poll jitter found in Notebook 01. This is acceptable for the MVP scope.)*

## Summary of Engineering Decisions

| Decision | Value | Source |
|---|---|---|
| Direction resolution | `trips.txt` join on `trip_id`: 98.5% resolved, no pipeline change needed | Section A |
| Bunching definition | same route + same direction + distance + persistence | Section B |
| Distance threshold | **100m** — operationally anchored (~5-8 vehicle lengths), no cliff nearby in the sweep | Section C |
| Persistence requirement | **2 consecutive polls (~30s)**, filters single-ping noise without over-restricting | Section C |
| Known limitation | 15s bucket flooring can split one real event into two if polls straddle a boundary | Section C |
| Follow-up for production | Add `direction_id` to `validator.ts` to drop the `trips.txt` runtime dependency | Section A |